# Train YOLO nhận diện biển báo VN trên Google Colab

Chuẩn bị: trên máy local chạy `split_dataset.py`, nén thư mục `dataset/` thành `dataset.zip` và upload lên Google Drive tại `MyDrive/vnsign/dataset.zip`.

Runtime → Change runtime type → GPU (A100 / L4 với Colab Pro).

In [ ]:
!nvidia-smi

In [ ]:
# Đổi URL thành repo GitHub của nhóm (repo private thì dùng token hoặc upload code lên Drive)
REPO_URL = 'https://github.com/<user>/<repo>.git'
!git clone -q $REPO_URL /content/project
%cd /content/project
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/vnsign'
# Giải nén vào ổ local của Colab (nhanh hơn nhiều so với đọc trực tiếp từ Drive)
!unzip -q -o $DRIVE/dataset.zip -d /content/project/
!python scripts/dataset_stats.py

## Train
Kết quả lưu thẳng lên Drive (`--project`) để không mất khi Colab ngắt kết nối. Mỗi thí nghiệm đặt một `--name` riêng.

In [ ]:
!python scripts/train.py --model yolo11s.pt --name s_960 --imgsz 960 --project $DRIVE/runs

In [ ]:
# Nếu Colab bị ngắt giữa chừng: train tiếp từ last.pt
# !python scripts/train.py --model $DRIVE/runs/s_960/weights/last.pt --name s_960 --resume

In [ ]:
from IPython.display import Image, display
RUN = f'{DRIVE}/runs/s_960'
for f in ['results.png', 'confusion_matrix_normalized.png', 'PR_curve.png', 'val_batch0_pred.jpg']:
    display(Image(filename=f'{RUN}/{f}', width=900))
!cat $RUN/test_summary.json

In [ ]:
# Chạy thử trên 1 video hành trình đã upload lên Drive
!python scripts/predict.py --weights $RUN/weights/best.pt --source $DRIVE/test_videos/route1.mp4 --out $DRIVE/outputs/route1